In [113]:
from BayesianFNN import BayesianFNN
import random
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from tqdm import tqdm
import torch.optim as optim
import torch.nn as nn
import copy
import pandas as pd
import importlib
import matplotlib.pyplot as plt

In [114]:
import BayesianFNN

importlib.reload(BayesianFNN)
from BayesianFNN import BayesianFNN  # re-import

In [115]:
# Set all random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

In [116]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Random seed set to: {SEED} for full reproducibility")

Using device: cpu
Random seed set to: 42 for full reproducibility


In [117]:
def seed_worker(worker_id):
    """Function to ensure DataLoader workers use different seeds derived from the base seed"""
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [118]:
transform = transforms.Compose([
    transforms.ToTensor(),  
    transforms.Lambda(lambda x: x.view(-1)) 
])

training_data = datasets.FashionMNIST(
    root="../../Datasets",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="../../Datasets",
    train=False,
    download=True,
    transform=transform
)

In [119]:
def plot_metrics(metrics_dict, save_path='./results/metrics_comparison.png'):
    """Plot comparison of metrics across all models"""
    # Define colors for each model
    colors = {
        'baseline': 'blue',
        'strong_baseline': 'yellow',
        'plasticity_multi_growth': 'red',
        'plasticity_single_growth': 'green'
    }
    
    # Create figure with subplots
    fig, axs = plt.subplots(4, 3, figsize=(20, 15))
    
    # Training loss total
    ax = axs[0, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_total']))
        ax.plot(epochs, metrics['train_loss_total'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (nll + kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Training loss nll
    ax = axs[0, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_nll']))
        ax.plot(epochs, metrics['train_loss_nll'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (nll)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Training loss kl
    ax = axs[0, 2]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_kl']))
        ax.plot(epochs, metrics['train_loss_kl'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Training Accuracy
    ax = axs[1, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_acc']))
        ax.plot(epochs, metrics['train_acc'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Accuracy')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()

    # Training Brier
    ax = axs[1, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_brier']))
        ax.plot(epochs, metrics['train_brier'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Brier')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Brier')
    ax.legend()
    
    # Validation loss total
    ax = axs[2, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_total']))
        ax.plot(epochs, metrics['val_loss_total'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (nll + kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Validation loss nll
    ax = axs[2, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_nll']))
        ax.plot(epochs, metrics['val_loss_nll'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (nll)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Validation loss kl
    ax = axs[2, 2]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_kl']))
        ax.plot(epochs, metrics['val_loss_kl'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()
    
    # Validation accuracy
    ax = axs[3, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_acc']))
        ax.plot(epochs, metrics['val_acc'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Accuracy')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()

    # Validation brier
    ax = axs[3, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_brier']))
        ax.plot(epochs, metrics['val_brier'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Brier')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Brier')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [120]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0.025, verbose=True):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False
    
    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [121]:
def loss_function(outputs, labels, kl_loss, beta=0.5):
    criterion = nn.CrossEntropyLoss()
    nll = criterion(outputs, labels)
    # normalise to per sample
    return nll + kl_loss*beta, nll, kl_loss*beta

In [122]:
def train(model, train_dataloader, optimizer, epoch, device, warmup_epochs=50):
    model.train()
    running_loss_total = 0.0
    running_loss_nll= 0.0
    running_loss_kl = 0.0
    running_brier = 0.0
    correct = 0
    total = 0
    beta = 1/len(train_dataloader.dataset)
    
    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch}')
    
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss, nll, kl = loss_function(outputs, labels, model.kl_loss(), beta = beta)
        loss.backward()
        optimizer.step()
        
        # Track statistics
        running_loss_total += loss.item()
        running_loss_nll += nll.item()
        running_loss_kl += kl.item()

        # Accuracy
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Brier Score
        probs = torch.softmax(outputs, dim=1)
        num_classes = outputs.size(1)
        one_hot = torch.nn.functional.one_hot(labels, num_classes=num_classes).float()
        
        brier = torch.sum((probs - one_hot) ** 2, dim=1).sum()
        running_brier += brier.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': running_loss_total / (progress_bar.n + 1),
            'acc': 100. * correct / total,
            'brier': running_brier / total
        })
    train_loss_total = running_loss_total / len(train_dataloader)
    train_acc = 100. * correct / total
    train_loss_nll = running_loss_nll / len(train_dataloader)
    train_loss_kl = running_loss_kl / len(train_dataloader)
    train_brier = running_brier / total
    
    return train_loss_total, train_acc, train_loss_nll, train_loss_kl, train_brier

In [123]:
def validate(model, val_dataloader, device):
    model.eval()
    val_loss_total = 0.0
    val_loss_nll = 0.0
    val_loss_kl = 0.0
    running_brier = 0.0
    correct = 0
    total = 0
    beta = 1/len(val_dataloader.dataset)

    with torch.no_grad():
        for inputs, labels in tqdm(val_dataloader, desc='Validating'):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss, nll, kl = loss_function(outputs, labels, model.kl_loss(), beta = beta)

            # Track Statistics
            val_loss_total += loss.item()
            val_loss_nll += loss.item()
            val_loss_kl += loss.item()
            
            # Accuracy
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Brier Score
            probs = torch.softmax(outputs, dim=1)
            num_classes = outputs.size(1)
            one_hot = torch.nn.functional.one_hot(labels, num_classes=num_classes).float()
            
            brier = torch.sum((probs - one_hot) ** 2, dim=1).sum()
            running_brier += brier.item()
            

    val_loss_total = val_loss_total / len(val_dataloader)
    val_loss_nll = val_loss_nll / len(val_dataloader)
    val_loss_kl = val_loss_kl / len(val_dataloader)
    val_acc = 100. * correct / total
    val_brier = running_brier / total
    
    return val_loss_total, val_acc, val_loss_nll, val_loss_kl, val_brier


In [124]:
def snr_based_neurogenesis(plasticity_original, hidden_sizes, neurons_to_add=16, exclude=[0]):
    snr = plasticity_original.get_average_snr_per_layer()
    print("\n Average Signal-to-Noise Ratio per Hidden Layer:")
    for i, val in enumerate(snr):
        print(f"  Layer {i+1}: {val.item():.4f}")
    layer_to_expand = min(
        (i for i in range(len(uncertainty)) if i not in exclude),
        key=lambda i: uncertainty[i]
    )
    print(f"Expanding Layer {layer_to_expand+1} "
          f"(lowest SNR: {snr[layer_to_expand].item():.4f}) "
          f"by {neurons_to_add} neurons")
    expanded_hidden_sizes = hidden_sizes.copy()
    expanded_hidden_sizes[layer_to_expand] += neurons_to_add
    plasticity_neurogenesis = BayesianFNN(784, expanded_hidden_sizes, 10).to(device)
    return plasticity_neurogenesis, expanded_hidden_sizes

In [125]:
def uncertainty_based_neurogenesis(plasticity_original, hidden_sizes, neurons_to_add=16, exclude=[0]):
    uncertainty = plasticity_original.get_average_uncertainty_per_layer()
    print("\n Average Uncertainty per Hidden Layer:")
    for i, val in enumerate(uncertainty):
        print(f"  Layer {i+1}: {val.item():.4f}")
    layer_to_expand = max(
        (i for i in range(len(uncertainty)) if i not in exclude),
        key=lambda i: uncertainty[i]
    )
    print(f"Expanding Layer {layer_to_expand+1} "
          f"(Highest Uncertainty: {uncertainty[layer_to_expand].item():.4f}) "
          f"by {neurons_to_add} neurons")
    expanded_hidden_sizes = hidden_sizes.copy()
    expanded_hidden_sizes[layer_to_expand] += neurons_to_add
    plasticity_neurogenesis = BayesianFNN(784, expanded_hidden_sizes, 10).to(device)
    return plasticity_neurogenesis, expanded_hidden_sizes

In [126]:
def expand_and_load_encoder_layer(old_sd, new_layer):
    new_sd = new_layer.state_dict()
    for k in new_sd.keys():
        if k not in old_sd:
            print(f"[skip] {k} not found in old layer")
            continue

        old_param = old_sd[k]
        new_param = new_sd[k]

        if old_param.shape == new_param.shape:
            new_sd[k] = old_param
        elif len(old_param.shape) == 2:
            # Linear weights: expand top-left corner
            new_sd[k][:old_param.shape[0], :old_param.shape[1]] = old_param
        elif len(old_param.shape) == 1:
            # Bias / LayerNorm
            new_sd[k][:old_param.shape[0]] = old_param
        else:
            print(f"[warn] Shape mismatch for {k}: old {old_param.shape}, new {new_param.shape}")

    new_layer.load_state_dict(new_sd, strict=True)

In [127]:
def snr_based_neuroapoptosis(plasticity_model, threshold=3, exclude=[0]):
    keep_dict  = {}
    print("\n Neurons Pruned from Each Hidden Layer:")
    for i, layer in enumerate(plasticity_model.layers):
        snr = layer.get_snr()
        snr_per_neuron = torch.mean(snr, dim=1)
        if i not in exclude:
            mask = snr_per_neuron >= threshold
            keep_dict[i] = mask.nonzero(as_tuple=True)[0].tolist()
        if i in exclude:
            keep_dict[i] = [i for i in range(len(snr_per_neuron))]
        print(f"Hidden Layer {i+1}: {len(snr_per_neuron)-len(keep_dict[i])}")
    return keep_dict 

In [128]:
def truncate_and_load_encoder_layer(old_sd, keep_dict, new_layer):
    num_layers = len(keep_dict)
    new_sd = {}
    for i in range(num_layers):
        keep_i = keep_dict.get(i, None)
        keep_prev = keep_dict.get(i - 1, None)
        for p in ["mu_w", "rho_w", "mu_b", "rho_b"]:
            key = f"layers.{i}.{p}"
            if key not in old_sd:
                continue
            w = old_sd[key]
            # weights (2D)
            if w.ndim == 2:
                if keep_i is not None:
                    w = w[keep_i, :]
                if keep_prev is not None:
                    w = w[:, keep_prev]
            # bias (1D)
            else:
                if keep_i is not None:
                    w = w[keep_i]
            new_sd[key] = w
    for p in ["mu_w", "rho_w", "mu_b", "rho_b"]:
        key = f"out.{p}"
        if key not in old_sd:
            continue
        w = old_sd[key]
        keep_last = keep_dict.get(num_layers - 1, None)
        if w.ndim == 2 and keep_last is not None:
            w = w[:, keep_last]
        new_sd[key] = w
    new_layer.load_state_dict(new_sd, strict=True)

In [129]:
def naive_truncate_and_load_encoder_layer(old_sd, new_layer):
    new_sd = new_layer.state_dict()

    new_trunc_sd = {}

    for k in new_sd.keys():
        if k not in old_sd:
            print(f"[skip] {k} not found in origin_layer")
            continue

        old_param = old_sd[k]
        new_param = new_sd[k]

        if old_param.shape == new_param.shape:
            new_trunc_sd[k] = old_param
        elif len(old_param.shape) == 2:
            # Linear weights
            new_trunc_sd[k] = old_param[:new_param.shape[0], :new_param.shape[1]]
        elif len(old_param.shape) == 1:
            # Biases / LayerNorm
            new_trunc_sd[k] = old_param[:new_param.shape[0]]
        else:
            print(f"[warn] {k} shape mismatch: old {old_param.shape}, new {new_param.shape}")
            continue

    new_layer.load_state_dict(new_trunc_sd, strict=True)

In [134]:
def run_experiment(experiment_name, model, train_loader, val_loader, test_loader, num_epochs, 
                   learning_rate=0.001, start_epoch=1, early_stopper=None, metrics=None, rewind=None):
    """Run a complete training experiment and return metrics"""
    print(f"\n{'-'*20} Running {experiment_name} experiment {'-'*20}")
    
    # Display model parameters
    param_stats = model.get_param_stats() if hasattr(model, 'get_param_stats') else {
        'total_params': sum(p.numel() for p in model.parameters()),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad)
    }
    
    print(f"Model parameters: {param_stats['total_params']:,}")
    print(f"Trainable parameters: {param_stats.get('trainable_params', param_stats['total_params']):,}")
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    rewind_state = None
    
    # Track metrics
    if not metrics:
        metrics = {}
        metrics['train_loss_total'] = []
        metrics['train_loss_nll'] = []
        metrics['train_loss_kl'] = []
        metrics['train_acc'] = []
        metrics['train_brier'] = []
        metrics['val_loss_total'] = []
        metrics['val_loss_nll'] = []
        metrics['val_loss_kl'] = []
        metrics['val_acc'] = []
        metrics['val_brier'] = []

    best_loss = 0.0
    best_model_state = None
    # Training loop
    for epoch in range(start_epoch, start_epoch + num_epochs):
        num_epochs = epoch
        # store rewind state
        if epoch - start_epoch == rewind:
            rewind_state = copy.deepcopy(model.state_dict())
        
        # Train
        train_loss_total, train_acc, train_loss_nll, train_loss_kl, train_brier = train(model, train_loader, optimizer, epoch, device)
        metrics['train_loss_total'].append(train_loss_total)
        metrics['train_loss_nll'].append(train_loss_nll)
        metrics['train_loss_kl'].append(train_loss_kl)
        metrics['train_acc'].append(train_acc)
        metrics['train_brier'].append(train_brier)
        
        # Validate
        val_loss_total, val_acc, val_loss_nll, val_loss_kl, val_brier = validate(model, val_loader, device)
        metrics['val_loss_total'].append(val_loss_total)
        metrics['val_loss_nll'].append(val_loss_nll)
        metrics['val_loss_kl'].append(val_loss_kl)
        metrics['val_acc'].append(val_acc)
        metrics['val_brier'].append(val_brier)
        
        print(f'Epoch {epoch}: Train Loss={train_loss_total:.4f}, Train Acc={train_acc:.2f}%, Train Brier={train_brier:.3f}, '
              f'Val Loss={val_loss_total:.4f}, Val Acc={val_acc:.2f}%, Val Brier={val_brier:.3f}')
        if val_loss_total < best_loss:
            best_loss = val_loss_total
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, f'./results/{experiment_name}/best_model.pth')
        
        if early_stopper:
            early_stopper.check_early_stop(val_loss_total)
            if early_stopper.stop_training:
                break

            

    # Plot and save metrics
    plot_metrics(
        {experiment_name: {
            'train_loss_total': metrics['train_loss_total'],
            'train_loss_nll': metrics['train_loss_nll'],
            'train_loss_kl': metrics['train_loss_kl'],
            'train_acc': metrics['train_acc'],
            'train_brier': metrics['train_brier'],
            'val_loss_total': metrics['val_loss_total'],
            'val_loss_nll': metrics['val_loss_nll'],
            'val_loss_kl': metrics['val_loss_kl'],
            'val_acc': metrics['val_acc'],
            'val_brier': metrics['val_brier']
        }}, 
        save_path=f'./results/{experiment_name}/metrics.png'
    )
    
    # Load best model for test
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model based on validation loss for final testing.")

    test_loss_total, test_acc, test_loss_nll, test_loss_kl, test_brier = validate(model, test_loader, device)

    print(f'Test Acc={test_acc:.2f}%, Test Loss={test_loss_total:.4f}, Test Brier={test_brier:.3f}')
    
    # Save model
    torch.save(model.state_dict(), f'./results/{experiment_name}/model.pth')
    
    # Update metrics
    metrics.update({
        'test_acc': test_acc,
        'test_loss_total': test_loss_total,
        'test_loss_nll': test_loss_nll,
        'test_loss_kl': test_loss_kl,
        'test_brier': test_brier,
        'param_count': param_stats['total_params'],
        'trainable_param_count': param_stats.get('trainable_params', param_stats['total_params']),
    })
    
    # Create a metrics DataFrame
    metrics_df = pd.DataFrame({
        'epoch': range(1, 1 + len(metrics['train_loss_total'])),
        'train_loss_total': metrics['train_loss_total'],
        'train_loss_nll': metrics['train_loss_nll'],
        'train_loss_kl': metrics['train_loss_kl'],
        'train_acc': metrics['train_acc'],
        'train_brier': metrics['train_brier'],
        'val_loss_total': metrics['val_loss_total'],
        'val_loss_nll': metrics['val_loss_nll'],
        'val_loss_kl': metrics['val_loss_kl'],
        'val_acc': metrics['val_acc'],
        'val_brier':metrics['val_brier']
    })
    metrics_df.to_csv(f'./results/{experiment_name}/metrics.csv', index=False)
    
    # Print summary
    print(f"\n{experiment_name} Summary:")
    print(f"Best validation accuracy: {max(metrics['val_acc'][start_epoch-1:]):.2f}%")
    print(f"Best validation loss: {min(metrics['val_loss_total'][start_epoch-1:]):.4f}")
    print(f"Best validation brier: {min(metrics['val_brier'][start_epoch-1:]):.3f}")
    print(f"Final test accuracy: {test_acc:.2f}%")
    print(f"Final test loss: {test_loss_total:.4f}%")
    print(f"Final test brier: {test_brier:.3f}")
    
    return metrics, model, num_epochs, rewind_state



In [145]:
def run_plasticity_experiment(
    experiment_name,
    base_model,
    hidden_sizes,
    train_loader,
    val_loader,
    test_loader,
    num_epochs,
    learning_rate,
    rewind_state_baseline=None,
    use_rewind=False,
    use_multi_growth=True,
    growth_epochs=20,
    neurons_to_add=2,
    prune_threshold=3,
    strong_baseline=False
):
    print("\n\n" + "="*50)
    print(f"Training {experiment_name.upper()}")
    print("="*50)

    # ===== Init =====
    plasticity_model = base_model
    if rewind_state_baseline is not None:
        plasticity_model.load_state_dict(rewind_state_baseline)

    genesis_hidden_sizes = hidden_sizes.copy()
    metrics = None
    num_epochs_used = 0

    print("+"*20 + " Growing Phase " + "+"*20)

    # =========================================================
    # CASE 1: MULTI-GROWTH 
    # =========================================================
    if use_multi_growth:
        prev_val_loss = float("inf")
        first_flag = True

        while True:
            remaining_epochs = max(0, num_epochs - num_epochs_used)
            if remaining_epochs == 0:
                break

            metrics, plasticity_model, num_epochs_used, rewind_state = run_experiment(
                experiment_name,
                plasticity_model,
                train_loader,
                val_loader,
                test_loader,
                remaining_epochs,
                learning_rate,
                start_epoch=1 + num_epochs_used,
                early_stopper=EarlyStopping(patience=3, delta=0.025),
                metrics=metrics,
                rewind=1 if (use_rewind and first_flag) else None
            )

            current_best_val_loss = min(metrics["val_loss_total"])

            # Stop if no improvement
            if current_best_val_loss >= prev_val_loss:
                break

            prev_val_loss = current_best_val_loss

            # Grow network 
            old_model = plasticity_model
            new_model, genesis_hidden_sizes = uncertainty_based_neurogenesis(
                old_model,
                genesis_hidden_sizes,
                neurons_to_add=neurons_to_add,
                exclude=[0]
            )

            if use_rewind:
                assert rewind_state is not None
                expand_and_load_encoder_layer(rewind_state, new_model)
            else:
                expand_and_load_encoder_layer(old_model.state_dict(), new_model)

            plasticity_model = new_model
            first_flag = False

    # =========================================================
    # CASE 2: SINGLE GROWTH 
    # =========================================================
    else:
        # -------------------------
        # Stage 1: Train BASE model
        # -------------------------
        base_epochs = min(growth_epochs, num_epochs)
        
        metrics, plasticity_model, num_epochs_used, rewind_state = run_experiment(
            experiment_name,
            plasticity_model,
            train_loader,
            val_loader,
            test_loader,
            base_epochs,
            learning_rate,
            start_epoch=1,
            metrics=metrics
        )
        
        # -------------------------
        # Stage 2: GROW
        # -------------------------
        old_model = plasticity_model
        new_model, genesis_hidden_sizes = uncertainty_based_neurogenesis(
            old_model,
            genesis_hidden_sizes,
            neurons_to_add=neurons_to_add,
            exclude=[0]
        )
        expand_and_load_encoder_layer(old_model.state_dict(), new_model)
    
        plasticity_model = new_model
    
        # -------------------------
        # Stage 3: Train GROWN model
        # -------------------------
        remaining_epochs = max(0, num_epochs - num_epochs_used)
        grow_train_epochs = min(growth_epochs, remaining_epochs)
    
        metrics, plasticity_model, num_epochs_used, _ = run_experiment(
            experiment_name,
            plasticity_model,
            train_loader,
            val_loader,
            test_loader,
            grow_train_epochs,
            learning_rate,
            start_epoch=1 + num_epochs_used,
            metrics=metrics
        )

    # =========================================================
    # PRUNING PHASE 
    # =========================================================
    print("-"*20 + " Pruning Phase " + "-"*20)

    if not strong_baseline:
        keep_dict = snr_based_neuroapoptosis(
            plasticity_model,
            threshold=prune_threshold,
            exclude=[0]
        )
    
        apoptosis_hidden_sizes = [len(keep_dict[i]) for i in range(len(keep_dict))]
    
        device = next(plasticity_model.parameters()).device
        new_model = BayesianFNN(784, apoptosis_hidden_sizes, 10).to(device)
    
        truncate_and_load_encoder_layer(plasticity_model.state_dict(), keep_dict, new_model)
    
        plasticity_model = new_model

    else:
        new_model = old_model
        naive_truncate_and_load_encoder_layer(plasticity_model.state_dict(), new_model)
        plasticity_model = new_model
        
    
    # =========================================================
    # FINAL TRAINING
    # =========================================================
    remaining_epochs = max(0, num_epochs - num_epochs_used)
    if remaining_epochs > 0:
        metrics, plasticity_model, num_epochs_used, _ = run_experiment(
            experiment_name,
            plasticity_model,
            train_loader,
            val_loader,
            test_loader,
            remaining_epochs,
            learning_rate,
            start_epoch=1 + num_epochs_used,
            metrics=metrics,
            rewind=None
        )

    return metrics, plasticity_model

In [146]:
def main():
    # Hyperparameters
    num_epochs = 200
    batch_size = 1024
    learning_rate = 0.01
    hidden_sizes = [12,12,12,12]
    rewind_state_baseline = None
    
    # Create results directory
    os.makedirs('results', exist_ok=True)
    

    # Create datasets
    transform = transforms.Compose([
        transforms.ToTensor(),  
        transforms.Lambda(lambda x: x.view(-1)) 
    ])
    
    training_data = datasets.FashionMNIST(
        root="../../Datasets",
        train=True,
        download=True,
        transform=transform
    )
    
    train_size = int(0.8 * len(training_data))
    val_size = len(training_data) - train_size 
    
    train_dataset, val_dataset = random_split(training_data, [train_size, val_size])

    test_dataset = datasets.FashionMNIST(
        root="../../Datasets",
        train=False,
        download=True,
        transform=transform
    )
    
    # Create data loaders with fixed seeds for workers
    g = torch.Generator()
    g.manual_seed(SEED)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=4,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    # ========== Experiment 1: Baseline Model ==========
    print("\n\n" + "="*50)
    print("Training Baseline Model")
    print("="*50)
    baseline_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    baseline_metrics, baseline_model, num_epochs_used, rewind_state_baseline = run_experiment(
        'baseline', 
        baseline_model, 
        train_loader, 
        val_loader, 
        test_loader, 
        num_epochs, 
        learning_rate,
        start_epoch=1,
        #early_stopper=EarlyStopping()
        rewind = 0
    )
    
    # ========== Experiment 2: Strong Baseline Model ==========
    base_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    strong_baseline_metrics, strong_baseline_model = run_plasticity_experiment(
        "strong_baseline",
        base_model,
        hidden_sizes,
        train_loader,
        val_loader,
        test_loader,
        num_epochs,
        learning_rate,
        rewind_state_baseline=rewind_state_baseline,
        use_rewind=False,
        use_multi_growth=False,
        growth_epochs=num_epochs//3,  
        neurons_to_add=12,
        prune_threshold=3,
        strong_baseline=True
    )

    # ========== Experiment 3: Multi-Growth  ==========
    
    base_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    plasticity_multi_growth_metrics, plasticity_multi_growth_model = run_plasticity_experiment(
        "plasticity_multi_growth",
        base_model,
        hidden_sizes,
        train_loader,
        val_loader,
        test_loader,
        num_epochs,
        learning_rate,
        rewind_state_baseline=rewind_state_baseline,
        use_rewind=False,
        use_multi_growth=True,
        growth_epochs=None,  
        neurons_to_add=1,
        prune_threshold=3,
    )

    # ========== Experiment 4: Single Growth ==========

    base_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    plasticity_single_growth_metrics, plasticity_single_growth_model = run_plasticity_experiment(
        "plasticity_single_growth",
        base_model,
        hidden_sizes,
        train_loader,
        val_loader,
        test_loader,
        num_epochs,
        learning_rate,
        rewind_state_baseline=rewind_state_baseline,
        use_rewind=False,
        use_multi_growth=False,
        growth_epochs=40,  
        neurons_to_add=12,
        prune_threshold=3,
    )
    
    # ========== Compare Results ==========
    # Combine all metrics
    all_metrics = {
        'baseline': baseline_metrics,
        'strong_baseline': strong_baseline_metrics,
        'plasticity_multi_growth': plasticity_multi_growth_metrics,
        'plasticity_single_growth': plasticity_single_growth_metrics
    }
    plot_metrics(all_metrics, save_path='./results/model_comparison.png')
    
    # Create summary table
    summary = pd.DataFrame([
        {
            'Model': 'Baseline',
            'Parameters': baseline_metrics['param_count'],
            'Trainable Params': baseline_metrics['trainable_param_count'],
            'Best Val Acc': max(baseline_metrics['val_acc']),
            'Best Val Brier': min(baseline_metrics['val_brier']),
            'Test Acc': baseline_metrics['test_acc'],
            'Test Brier': baseline_metrics['test_brier'],
        },
        {
            'Model': 'Strong Baseline',
            'Parameters': strong_baseline_metrics['param_count'],
            'Trainable Params': strong_baseline_metrics['trainable_param_count'],
            'Best Val Acc': max(strong_baseline_metrics['val_acc']),
            'Best Val Brier': min(strong_baseline_metrics['val_brier']),
            'Test Acc': strong_baseline_metrics['test_acc'],
            'Test Brier': strong_baseline_metrics['test_brier'],
        },
        {
            'Model': 'Plasticity Multi Growth',
            'Parameters': plasticity_multi_growth_metrics['param_count'],
            'Trainable Params': plasticity_multi_growth_metrics['trainable_param_count'],
            'Best Val Acc': max(plasticity_multi_growth_metrics['val_acc']),
            'Best Val Brier': min(plasticity_multi_growth_metrics['val_brier']),
            'Test Acc': plasticity_multi_growth_metrics['test_acc'],
            'Test Brier': plasticity_multi_growth_metrics['test_brier'],
        },
        {
            'Model': 'Plasticity Single Growth',
            'Parameters': plasticity_single_growth_metrics['param_count'],
            'Trainable Params': plasticity_single_growth_metrics['trainable_param_count'],
            'Best Val Acc': max(plasticity_single_growth_metrics['val_acc']),
            'Best Val Brier': min(plasticity_single_growth_metrics['val_brier']),
            'Test Acc': plasticity_single_growth_metrics['test_acc'],
            'Test Brier': plasticity_single_growth_metrics['test_brier'],
        }
    ])
    
    summary.to_csv('./results/experiment_summary.csv', index=False)
    print("\nExperiment Summary:")
    print(summary)
    return baseline_model, strong_baseline_model, plasticity_multi_growth_model, plasticity_single_growth_model

In [ ]:
baseline_model, strong_baseline_model, plasticity_multi_growth_model, plasticity_single_growth_model = main()



Training Baseline Model

-------------------- Running baseline experiment --------------------
Model parameters: 20,036
Trainable parameters: 20,036


Validating: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 31.02it/s]


Epoch 1: Train Loss=2.3727, Train Acc=24.42%, Train Brier=0.824, Val Loss=3.3397, Val Acc=37.12%, Val Brier=0.732


Validating: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 31.51it/s]


Epoch 2: Train Loss=1.7426, Train Acc=44.67%, Train Brier=0.662, Val Loss=2.9307, Val Acc=46.00%, Val Brier=0.634


...

Validating: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 31.80it/s]


Epoch 106: Train Loss=0.6995, Train Acc=81.42%, Train Brier=0.267, Val Loss=1.0804, Val Acc=83.37%, Val Brier=0.233


Validating: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 31.13it/s]


Epoch 107: Train Loss=0.6246, Train Acc=84.27%, Train Brier=0.224, Val Loss=1.0945, Val Acc=83.45%, Val Brier=0.240


Validating:   8%|██████                                                                  | 1/12 [00:00<00:02,  5.08it/s]